# 🏥 معالجة دفعية للـ OCR الطبي + تصحيح تفاعلي + تصدير HuggingFace

دفتر مُتكامل لمعالجة ملفات PDF الطبية دفعيًا مع تصحيح تفاعلي وتصدير إلى HuggingFace.

### الخلايا الأربع:
1. **الإعداد**: تثبيت الحزم وتهيئة البيئة
2. **المحرك الأساسي**: فئة `BatchMedicalOCR` للمعالجة
3. **واجهة التصحيح**: واجهة Gradio للمراجعة التفاعلية
4. **التدريب والتصدير**: بناء البيانات والتدريب والرفع إلى HF

## الخلية ١: الإعداد — تثبيت الحزم والتهيئة

In [ ]:
# ══════════════════════════════════════════════════════════════
# تثبيت جميع الحزم اللازمة
# ══════════════════════════════════════════════════════════════

# حزم OCR والتعلم العميق
!pip install -q transformers datasets accelerate torch pillow opencv-python-headless

# حزم معالجة PDF
!pip install -q fitz  # PyMuPDF

# حزم واجهة المستخدم
!pip install -q gradio

# حزم تصدير HuggingFace
!pip install -q huggingface_hub

# حزم التقييم
!pip install -q jiwer

print("✅ تمّ تثبيت جميع الحزم بنجاح")

# ── التحقق من التثبيت ──
import torch
import fitz  # PyMuPDF
import gradio as gr
import cv2
import numpy as np
from PIL import Image
from huggingface_hub import HfApi
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ PyMuPDF: {fitz.version}")
print(f"✅ GPU متاح: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

## الخلية ٢: المحرك الأساسي — فئة `BatchMedicalOCR`

In [ ]:
import os
import gc
import json
import re
import time
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field, asdict
from datetime import datetime

import fitz  # PyMuPDF
import cv2
import numpy as np
from PIL import Image
import torch
from transformers import TrOCRProcessor, VisionEncoderDecoderModel


# ══════════════════════════════════════════════════════════════
# قاموس المصطلحات الطبية للحماية
# ══════════════════════════════════════════════════════════════

MEDICAL_LEXICON = {
    # أدوية شائعة
    "أموكسيسيلين", "ميتفورمين", "أنسولين", "أوميبرازول", "أسبرين",
    "ليزينوبريل", "آتورفاستاتين", "بايريكسامول", "أموكسيل", "سيبروفلوكساسين",
    "ديكلوفيناك", "إيبوبروفين", "باراسيتامول", "كيتوبروفين", "نابروكسين",
    # تشخيصات
    "السكري", "ضغط الدم", "الربو", "التهاب", "ارتفاع", "انخفاض",
    "القلب", "الكبد", "الكلى", "المفاصل", "الجيوب الأنفية",
    # فحوصات
    "صورة دم", "تخطيط قلب", "ECG", "HbA1c", "TSH",
    "وظائف الكبد", "وظائف الكلى", "صورة الصدر",
    # وحدات
    "ملغ", "وحدات", "مم زئبق", "ملغ/ديسيلتر",
}


# ══════════════════════════════════════════════════════════════
# فئة المعالجة الدفعية
# ══════════════════════════════════════════════════════════════

@dataclass
class LineResult:
    """نتيجة سطر واحد من OCR."""
    page_num: int
    line_idx: int
    image_path: str = ""
    raw_text: str = ""
    corrected_text: str = ""
    confidence: float = 0.0


class BatchMedicalOCR:
    """فئة المعالجة الدفعية للـ OCR الطبي.

    توفّر:
    - تحويل PDF إلى صور (PyMuPDF)
    - معالجة مسبقة (إزالة الخربشة + CLAHE)
    - تقسيم الصفحة إلى أسطر
    - التعرف على النصوص (TrOCR + حماية المصطلحات الطبية)
    - استخراج المراجع الطبية
    - معالجة مجلد كامل
    """

    def __init__(
        self,
        model_name: str = "microsoft/trocr-base-handwritten",
        device: Optional[str] = None,
    ):
        # تحديد الجهاز
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        # تحميل النموذج والمعالج
        print(f"⏳ جاري تحميل النموذج: {model_name}")
        self.processor = TrOCRProcessor.from_pretrained(model_name)
        self.model = VisionEncoderDecoderModel.from_pretrained(model_name)
        self.model.eval()

        # تكوين النموذج
        self.model.config.decoder_start_token_id = self.processor.tokenizer.cls_token_id
        self.model.config.pad_token_id = self.processor.tokenizer.pad_token_id

        if self.device == "cuda":
            self.model = self.model.to(self.device)

        # قائمة النتائج
        self.results: List[LineResult] = []
        # قائمة صور الأسطر (للعرض)
        self.line_images: List[np.ndarray] = []

        print(f"✅ تمّ تهيئة المحرك على {self.device}")

    # ─────────────────────────────────────────────────────────
    # تحويل PDF إلى صور
    # ─────────────────────────────────────────────────────────
    def pdf_to_images(
        self,
        pdf_path: str,
        dpi: int = 300,
    ) -> List[Tuple[int, np.ndarray]]:
        """تحويل ملف PDF إلى قائمة صور.

        Args:
            pdf_path: مسار ملف PDF.
            dpi: دقة الصور الناتجة.

        Returns:
            قائمة (رقم_الصفحة، صورة_الصفحة).
        """
        # فتح ملف PDF باستخدام PyMuPDF
        doc = fitz.open(pdf_path)
        pages = []

        for page_num in range(len(doc)):
            page = doc.load_page(page_num)

            # تحويل الصفحة إلى صورة بدقة محددة
            mat = fitz.Matrix(dpi / 72, dpi / 72)
            pix = page.get_pixmap(matrix=mat)

            # تحويل إلى numpy array (RGB)
            img_data = pix.samples
            img_array = np.frombuffer(img_data, dtype=np.uint8)
            img_array = img_array.reshape((pix.height, pix.width, pix.n))

            # التحقق من عدد القنوات
            if pix.n == 4:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_RGBA2RGB)
            elif pix.n == 1:
                img_array = cv2.cvtColor(img_array, cv2.COLOR_GRAY2RGB)

            pages.append((page_num + 1, img_array))

        doc.close()
        print(f"✅ تمّ تحويل PDF إلى {len(pages)} صفحة")
        return pages

    # ─────────────────────────────────────────────────────────
    # المعالجة المسبقة
    # ─────────────────────────────────────────────────────────
    def preprocess(self, image: np.ndarray) -> np.ndarray:
        """معالجة الصورة مسبقًا — إزالة الخربشة + CLAHE.

        الخطوات:
            1. تحويل إلى رمادي
            2. إزالة الخطوط الأفقية والعمودية
            3. تطبيق CLAHE لتحسين التباين

        Args:
            image: صورة الإدخال.

        Returns:
            صورة معالجة.
        """
        # تحويل إلى رمادي
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image.copy()

        # عتبة ثنائية
        _, binary = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # ── إزالة الخطوط الأفقية ──
        horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (50, 1))
        horiz_lines = cv2.morphologyEx(
            binary, cv2.MORPH_OPEN, horiz_kernel, iterations=2
        )
        no_horiz = cv2.subtract(binary, horiz_lines)

        # ── إزالة الخطوط العمودية ──
        vert_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 50))
        vert_lines = cv2.morphologyEx(
            binary, cv2.MORPH_OPEN, vert_kernel, iterations=2
        )
        cleaned = cv2.subtract(no_horiz, vert_lines)

        # ── تطبيق CLAHE (تحسين التباين التكيفي) ──
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        enhanced = clahe.apply(gray)

        # دمج القناع المنظّف مع الصورة المحسّنة
        final = cv2.bitwise_and(enhanced, enhanced, mask=cleaned)

        return final

    # ─────────────────────────────────────────────────────────
    # تقسيم الأسطر
    # ─────────────────────────────────────────────────────────
    def segment_lines(self, image: np.ndarray) -> List[np.ndarray]:
        """تقسيم الصفحة إلى أسطر نصية.

        Args:
            image: صورة الصفحة.

        Returns:
            قائمة بصور الأسطر.
        """
        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        else:
            gray = image.copy()

        _, binary = cv2.threshold(
            gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # الإسقاط الأفقي
        h_proj = np.sum(binary, axis=1)

        lines = []
        in_line = False
        start_y = 0
        min_height = 8

        for y, count in enumerate(h_proj):
            if count > 0 and not in_line:
                in_line = True
                start_y = y
            elif count == 0 and in_line:
                in_line = False
                if y - start_y >= min_height:
                    lines.append((start_y, y))

        if in_line and len(h_proj) - start_y >= min_height:
            lines.append((start_y, len(h_proj)))

        margin = 3
        line_images = []
        for y1, y2 in lines:
            top = max(0, y1 - margin)
            bottom = min(image.shape[0], y2 + margin)
            line_img = image[top:bottom, :]
            line_images.append(line_img)

        return line_images

    # ─────────────────────────────────────────────────────────
    # التعرف على سطر واحد مع حماية المصطلحات الطبية
    # ─────────────────────────────────────────────────────────
    def ocr_line(
        self,
        line_image: np.ndarray,
    ) -> Tuple[str, float]:
        """التعرف على النص في سطر واحد.

        يستخدم TrOCR مع حماية المصطلحات الطبية
        لمنع تحريف أسماء الأدوية والتشخيصات.

        Args:
            line_image: صورة السطر.

        Returns:
            (النص المتعرف عليه، الثقة)
        """
        # تحويل إلى RGB
        if len(line_image.shape) == 2:
            rgb = cv2.cvtColor(line_image, cv2.COLOR_GRAY2RGB)
        else:
            rgb = line_image

        # تحويل إلى PIL
        pil_img = Image.fromarray(rgb)

        # معالجة بالـ processor
        pixel_values = self.processor(
            pil_img, return_tensors="pt"
        ).pixel_values

        if self.device == "cuda":
            pixel_values = pixel_values.to(self.device)

        # التوليد
        with torch.no_grad():
            generated = self.model.generate(pixel_values)
            text = self.processor.tokenizer.batch_decode(
                generated, skip_special_tokens=True
            )[0]

        # ── حماية المصطلحات الطبية ──
        protected_text = self._protect_medical_terms(text)

        # تقدير الثقة (قيمة تقريبية)
        confidence = 0.85 if text.strip() else 0.0

        return protected_text, confidence

    def _protect_medical_terms(self, text: str) -> str:
        """حماية المصطلحات الطبية من التشويه.

        يقارن كلمات النص مع القاموس الطبي ويصحّح
        الكلمات القريبة شبهيًا.

        Args:
            text: النص المعترف به.

        Returns:
            النص بعد حماية المصطلحات.
        """
        words = text.split()
        corrected = []

        for word in words:
            best_match = word
            best_dist = 2  # أقل مسافة مقبولة

            for term in MEDICAL_LEXICON:
                # حساب المسافة (عدد عمليات التعديل البسيطة)
                dist = self._edit_distance(word, term)
                if dist < best_dist and dist < len(term) * 0.3:
                    best_match = term
                    best_dist = dist

            corrected.append(best_match)

        return ' '.join(corrected)

    @staticmethod
    def _edit_distance(s1: str, s2: str) -> int:
        """حساب مسافة ليفنشتاين بين نصين."""
        if len(s1) < len(s2):
            return BatchMedicalOCR._edit_distance(s2, s1)
        if len(s2) == 0:
            return len(s1)

        prev_row = list(range(len(s2) + 1))
        for i, c1 in enumerate(s1):
            curr_row = [i + 1]
            for j, c2 in enumerate(s2):
                insertions = prev_row[j + 1] + 1
                deletions = curr_row[j] + 1
                substitutions = prev_row[j] + (c1 != c2)
                curr_row.append(min(insertions, deletions, substitutions))
            prev_row = curr_row

        return prev_row[-1]

    # ─────────────────────────────────────────────────────────
    # استخراج المراجع الطبية
    # ─────────────────────────────────────────────────────────
    def extract_refs(self, text: str) -> List[str]:
        """استخراج المراجع والأرقام الطبية من النص.

        يبحث عن:
            - أرقام الملفات
            - التواريخ
            - أرقام الهاتف
            - المراجع العلمية

        Args:
            text: النص الكامل.

        Returns:
            قائمة بالمراجع المستخرجة.
        """
        refs = []

        # نمط التواريخ (dd/mm/yyyy أو dd-mm-yyyy)
        date_pattern = r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b'
        refs.extend(re.findall(date_pattern, text))

        # نمط أرقام الملفات
        file_pattern = r'\b(?:ملف|رقم|File|No)\s*[:#]?\s*(\d+)\b'
        refs.extend(re.findall(file_pattern, text))

        # نمط أرقام الهاتف
        phone_pattern = r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b'
        refs.extend(re.findall(phone_pattern, text))

        return refs

    # ─────────────────────────────────────────────────────────
    # معالجة مجلد كامل
    # ─────────────────────────────────────────────────────────
    def process_folder(
        self,
        folder_path: str,
        output_json: str = "ocr_results.json",
    ) -> List[Dict]:
        """معالجة جميع ملفات PDF في مجلد.

        Args:
            folder_path: مسار المجلد.
            output_json: مسار ملف JSON الناتج.

        Returns:
            قائمة بنتائج OCR.
        """
        folder = Path(folder_path)
        pdf_files = list(folder.glob("**/*.pdf"))
        img_files = list(folder.glob("**/*.png")) + list(folder.glob("**/*.jpg"))

        all_files = pdf_files + img_files
        print(f"📁 وجد {len(all_files)} ملف في: {folder_path}")

        self.results = []
        self.line_images = []
        total_lines = 0

        for file_idx, file_path in enumerate(all_files):
            print(f"\n📄 معالجة ({file_idx+1}/{len(all_files)}): {file_path.name}")

            # ── تحميل الصور ──
            if file_path.suffix.lower() == '.pdf':
                pages = self.pdf_to_images(str(file_path))
            else:
                img = cv2.imread(str(file_path))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                pages = [(1, img)]

            # ── معالجة كل صفحة ──
            for page_num, page_img in pages:
                # المعالجة المسبقة
                processed = self.preprocess(page_img)

                # تقسيم الأسطر
                lines = self.segment_lines(processed)

                # التعرف على كل سطر
                for line_idx, line_img in enumerate(lines):
                    text, conf = self.ocr_line(line_img)

                    result = LineResult(
                        page_num=page_num,
                        line_idx=line_idx,
                        raw_text=text,
                        corrected_text=text,  # النسخة الأولية
                        confidence=conf,
                    )

                    self.results.append(result)
                    self.line_images.append(line_img)
                    total_lines += 1

        # ── حفظ النتائج ──
        results_data = [asdict(r) for r in self.results]
        with open(output_json, 'w', encoding='utf-8') as f:
            json.dump(results_data, f, ensure_ascii=False, indent=2)

        print(f"\n✅ تمّ معالجة {total_lines} سطر من {len(all_files)} ملف")
        print(f"📁 النتائج محفوظة في: {output_json}")

        return results_data


# ── إنشاء نسخة من المحرك ──
print("⏳ جاري تهيئة محرك OCR الطبي...")
ocr_engine = BatchMedicalOCR()
print("✅ المحرك جاهز للاستخدام")

## الخلية ٣: واجهة التصحيح التفاعلية — Gradio

In [ ]:
import gradio as gr
import json


# ══════════════════════════════════════════════════════════════
# حالة واجهة التصحيح
# ══════════════════════════════════════════════════════════════

class CorrectionUIState:
    """حالة واجهة التصحيح التفاعلية."""
    def __init__(self):
        self.current_idx = 0
        self.corrections: List[str] = []
        self.loaded = False

ui_state = CorrectionUIState()


def load_results_json(file_obj):
    """تحميل نتائج OCR من ملف JSON.

    Args:
        file_obj: كائن الملف المُحمَّل.

    Returns:
        (صورة، نص خام، نص مُصحَّح، معلومات)
    """
    if file_obj is None:
        blank = np.zeros((100, 600, 3), dtype=np.uint8)
        return blank, "", "", "حمّل ملف JSON أولاً"

    file_path = file_obj.name if hasattr(file_obj, 'name') else file_obj

    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # تحميل البيانات في واجهة التصحيح
    ui_state.corrections = [item.get("corrected_text", item.get("raw_text", "")) for item in data]
    ui_state.current_idx = 0
    ui_state.loaded = True

    return get_current_item()


def get_current_item():
    """جلب العنصر الحالي للعرض."""
    if not ui_state.loaded or not ui_state.corrections:
        blank = np.zeros((100, 600, 3), dtype=np.uint8)
        return blank, "", "", "لا توجد بيانات"

    idx = ui_state.current_idx
    total = len(ui_state.corrections)

    # صورة السطر (إذا كانت متوفرة)
    if ocr_engine.line_images and idx < len(ocr_engine.line_images):
        img = ocr_engine.line_images[idx]
        if len(img.shape) == 2:
            display_img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        else:
            display_img = img
    else:
        display_img = np.zeros((64, 600, 3), dtype=np.uint8)

    if idx < len(ocr_engine.results):
        raw = ocr_engine.results[idx].raw_text
    else:
        raw = ""

    corrected = ui_state.corrections[idx]
    info = f"سطر {idx + 1} من {total}"

    return display_img, raw, corrected, info


def save_and_next_ui(corrected_text):
    """حفظ التصحيح والانتقال للتالي."""
    if not ui_state.loaded:
        return get_current_item()

    ui_state.corrections[ui_state.current_idx] = corrected_text
    if ui_state.current_idx < len(ui_state.corrections) - 1:
        ui_state.current_idx += 1

    return get_current_item()


def save_and_prev_ui(corrected_text):
    """حفظ التصحيح والعودة للسابق."""
    if not ui_state.loaded:
        return get_current_item()

    ui_state.corrections[ui_state.current_idx] = corrected_text
    if ui_state.current_idx > 0:
        ui_state.current_idx -= 1

    return get_current_item()


def save_all_corrections():
    """حفظ جميع التصحيحات في ملف JSON."""
    if not ui_state.loaded:
        return "لا توجد تصحيحات لحفظها"

    # تحديث النتائج بالتصحيحات
    for i, corr in enumerate(ui_state.corrections):
        if i < len(ocr_engine.results):
            ocr_engine.results[i].corrected_text = corr

    output_path = "/content/corrected_results.json"
    results_data = [asdict(r) for r in ocr_engine.results]
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(results_data, f, ensure_ascii=False, indent=2)

    return f"✅ تمّ حفظ {len(ui_state.corrections)} تصحيح في: {output_path}"


# ══════════════════════════════════════════════════════════════
# بناء واجهة Gradio
# ══════════════════════════════════════════════════════════════

with gr.Blocks(title="تصحيح OCR الطبي", theme=gr.themes.Soft()) as correction_ui:
    gr.Markdown("# ✏️ تصحيح OCR الطبي — واجهة تفاعلية")
    gr.Markdown("حمّل ملف JSON الناتج من المعالجة الدفعية، ثم راجع وصحّح كل سطر.")

    # تحميل الملف
    with gr.Row():
        json_input = gr.File(label="📁 رفع ملف JSON", file_types=[".json"])
        load_btn = gr.Button("📂 تحميل النتائج", variant="primary")

    info_display = gr.Textbox(label="📌 المعلومات", interactive=False)

    with gr.Row():
        with gr.Column(scale=2):
            line_display = gr.Image(label="🖼️ صورة السطر", type="numpy")
        with gr.Column(scale=2):
            raw_display = gr.Textbox(label="🤖 النص الخام (OCR)", interactive=False, lines=3)
            corr_input = gr.Textbox(label="✏️ التصحيح", interactive=True, lines=3, placeholder="أدخل التصحيح...")

    with gr.Row():
        prev_btn = gr.Button("⬅️ السابق")
        next_btn = gr.Button("➡️ التالي")

    save_btn = gr.Button("💾 حفظ جميع التصحيحات", variant="secondary")
    save_status = gr.Textbox(label="📝 حالة الحفظ", interactive=False)

    # ربط الأحداث
    load_btn.click(
        fn=load_results_json,
        inputs=[json_input],
        outputs=[line_display, raw_display, corr_input, info_display],
    )
    next_btn.click(
        fn=save_and_next_ui,
        inputs=[corr_input],
        outputs=[line_display, raw_display, corr_input, info_display],
    )
    prev_btn.click(
        fn=save_and_prev_ui,
        inputs=[corr_input],
        outputs=[line_display, raw_display, corr_input, info_display],
    )
    save_btn.click(
        fn=save_all_corrections,
        inputs=[],
        outputs=[save_status],
    )


print("✅ تمّ بناء واجهة التصحيح التفاعلية")

## الخلية ٤: بناء البيانات + التدريب + التصدير إلى HuggingFace

In [ ]:
import zipfile
import shutil
from huggingface_hub import HfApi, create_repo


# ══════════════════════════════════════════════════════════════
# بناء مجموعة البيانات للتربية
# ══════════════════════════════════════════════════════════════

def build_training_dataset(
    results: List[LineResult],
    line_images: List[np.ndarray],
    output_dir: str = "./training_dataset",
):
    """تحويل النتائج المُصحَّحة إلى مجموعة بيانات للتربية.

    هيكل الإخراج:
        training_dataset/
        ├── images/
        │   ├── page001_line001.png
        │   └── ...
        └── labels.txt

    Args:
        results: قائمة نتائج OCR.
        line_images: قائمة صور الأسطر.
        output_dir: مجلد الإخراج.
    """
    out_path = Path(output_dir)
    images_dir = out_path / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    labels = []
    count = 0

    for i, (result, line_img) in enumerate(zip(results, line_images)):
        # تخطي الأسطر الفارغة
        if not result.corrected_text.strip():
            continue

        # اسم الصورة
        img_name = f"page{result.page_num:03d}_line{result.line_idx:03d}.png"

        # تحويل إلى RGB
        if len(line_img.shape) == 2:
            rgb_img = cv2.cvtColor(line_img, cv2.COLOR_GRAY2RGB)
        else:
            rgb_img = line_img

        # حفظ الصورة
        cv2.imwrite(str(images_dir / img_name), cv2.cvtColor(rgb_img, cv2.COLOR_RGB2BGR))

        # إضافة التصنيف
        labels.append(f"{img_name}\t{result.corrected_text}")
        count += 1

    # حفظ ملف التصنيفات
    labels_file = out_path / "labels.txt"
    labels_file.write_text("\n".join(labels), encoding="utf-8")

    print(f"✅ تمّ بناء مجموعة البيانات: {count} عينة في {output_dir}")
    return output_dir


# ══════════════════════════════════════════════════════════════
# تصدير إلى HuggingFace
# ══════════════════════════════════════════════════════════════

def export_to_huggingface(
    model_dir: str,
    repo_name: str,
    token: Optional[str] = None,
    private: bool = False,
):
    """رفع النموذج إلى HuggingFace Hub.

    Args:
        model_dir: مسار مجلد النموذج.
        repo_name: اسم المستودع (مثال: username/trocr-arabic-medical).
        token: رمز الوصول لـ HuggingFace.
        private: هل المستودع خاص؟
    """
    if token is None:
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            pass

    if not token:
        print("⚠️  لم يُعثر على رمز HF_TOKEN. اضبطه في Secrets أولاً.")
        return

    # إنشاء المستودع إذا لم يكن موجودًا
    api = HfApi()
    try:
        create_repo(
            repo_id=repo_name,
            token=token,
            private=private,
            exist_ok=True,
        )
        print(f"✅ تمّ إنشاء/التحقق من المستودع: {repo_name}")
    except Exception as e:
        print(f"⚠️  خطأ في إنشاء المستودع: {e}")
        return

    # رفع المجلد
    print(f"⏳ جاري رفع النموذج إلى {repo_name}...")
    api.upload_folder(
        folder_path=model_dir,
        repo_id=repo_name,
        token=token,
        commit_message="رفع نموذج OCR الطبي المُدرب",
    )

    print(f"✅ تمّ رفع النموذج بنجاح إلى:")
    print(f"   https://huggingface.co/{repo_name}")


print("✅ تمّ تعريف دوال بناء البيانات والتصدير")

## تشغيل: معالجة مجلد + التصحيح + التصدير

In [ ]:
# ══════════════════════════════════════════════════════════════
# مثال: معالجة مجلد من الملفات
# ══════════════════════════════════════════════════════════════

# ── إنشاء مجلد بيانات تجريبي ──
os.makedirs("/content/sample_pdfs", exist_ok=True)

# إنشاء صور تجريبية للاختبار
from PIL import ImageDraw, ImageFont

sample_texts = [
    "المريض يعاني من ارتفاع ضغط الدم",
    "الجرعة: 500 ملغ مرتين يوميا",
    "أموكسيسيلين 250 ملغ كل 8 ساعات",
    "تشخيص: التهاب الجيوب الأنفية المزمن",
    "مراجعة بعد شهر للاطمئنان",
]

for i, text in enumerate(sample_texts):
    img = Image.new("RGB", (640, 64), (255, 255, 255))
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype(
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 24
        )
    except Exception:
        font = ImageFont.load_default()
    draw.text((10, 15), text, fill=(0, 0, 0), font=font)
    img.save(f"/content/sample_pdfs/page_{i+1}.png")

print("✅ تمّ إنشاء بيانات تجريبية")

# ── معالجة المجلد ──
results = ocr_engine.process_folder(
    folder_path="/content/sample_pdfs",
    output_json="/content/ocr_results.json",
)

print(f"\n📊 تمّ استخراج {len(results)} سطر")
if results:
    print(f"📝 مثال: {results[0]['raw_text']}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# بناء مجموعة البيانات للتربية
# ══════════════════════════════════════════════════════════════

dataset_dir = build_training_dataset(
    results=ocr_engine.results,
    line_images=ocr_engine.line_images,
    output_dir="./training_dataset",
)

# عرض محتوى ملف التصنيفات
labels_file = Path("./training_dataset/labels.txt")
if labels_file.exists():
    content = labels_file.read_text(encoding='utf-8')
    lines = content.strip().split('\n')
    print(f"\n📄 ملف labels.txt — {len(lines)} سطر:")
    for line in lines[:5]:
        print(f"  {line}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# تصدير البيانات كـ ZIP
# ══════════════════════════════════════════════════════════════

zip_path = "/content/medical_ocr_training_data.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    dataset_path = Path("./training_dataset")
    for file_path in dataset_path.rglob("*"):
        if file_path.is_file():
            arcname = file_path.relative_to(dataset_path.parent)
            zf.write(str(file_path), str(arcname))

zip_size = os.path.getsize(zip_path) / (1024 * 1024)
print(f"✅ تمّ تصدير البيانات: {zip_path} ({zip_size:.2f} MB)")

In [ ]:
# ══════════════════════════════════════════════════════════════
# تشغيل واجهة التصحيح التفاعلية
# ══════════════════════════════════════════════════════════════

# ملاحظة: عدّل HF_TOKEN و repo_name قبل التشغيل
# export_to_huggingface(
#     model_dir="./training_dataset",
#     repo_name="your-username/trocr-arabic-medical",
# )

# تشغيل واجهة التصحيح
correction_ui.launch(share=True, debug=True)